# Transformations in Spark — select, filter, withColumn

Writing your first transformations with Spark DataFrame APIs
Yesterday, we learned how to create and explore DataFrames from CSV, JSON, and Parquet. We peeked at schemas, counted rows, and did some simple exploration. But exploration alone won’t get us far.

As data engineers, our real job is to transform data: clean it, reshape it, and prepare it for downstream analytics. That’s where Spark’s DataFrame transformations come in.

Today, we’ll focus on three of the most common transformations you’ll use every single day:

- select()
- filter()
- withColumn()

By the end, you’ll feel comfortable writing your very first Spark pipelines.

The Dataset We’ll Use
Let’s stick to the same sales.csv file from Day 6 for continuity.

/Volumes/thedataengineering_00/data/sampledata/sampleimages/sales.csv

region,revenue,date

North,1000,2025-09-01

South,1500,2025-09-01

East,1200,2025-09-01

West,,2025-09-01

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("day7-demo").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/sampleimages/sales.csv")
df = df.na.fill({"revenue": 0})  # replace missing revenue with 0
df.show()

Transformation 1: select()

The select method is like picking columns from a table.

In [0]:
# Select specific columns
df.select("region", "revenue").show()

In [0]:
from pyspark.sql.functions import col

df.select(col("region"), (col("revenue") * 1.1).alias("revenue_with_tax")).show()

Notice how we didn’t need to write loops — Spark applies the transformation across the dataset in parallel.

### Transformation 2: filter()
Filtering means keeping only the rows that satisfy a condition.

In [0]:
# Simple filter
df.filter(df["revenue"] > 1000).show()

In [0]:
df.filter((df["revenue"] > 1000) & (df["region"] != "South")).show()

### Transformation 3: withColumn()
The withColumn method lets you create a new column or replace an existing one

In [0]:
from pyspark.sql.functions import col

# Add new column with 10% discount
df_new = df.withColumn("discounted_revenue", col("revenue") * 0.9)
df_new.show()

In [0]:
from pyspark.sql.functions import to_date

# Convert string date to proper DateType
df_date = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
df_date.printSchema()

### Chaining Transformations
The real beauty is when you chain transformations together.

In [0]:
result = (
    df.withColumn("revenue_with_tax", col("revenue") * 1.05)
      .filter(col("revenue_with_tax") > 1100)
      .select("region", "revenue_with_tax")
)

result.show()

In just a few lines, we’ve:

- Added a new column,
- Applied a filter,
- Selected only the columns we need.

That’s a complete mini ETL step right there.

### A Mental Picture
Think of a DataFrame like a conveyor belt in a factory.

select is like choosing which parts to keep on the belt.
filter is like removing defective items as they pass.
withColumn is like adding a new label or tag to each item.

Each step adds to the plan. Only when you finally “ship” the belt (by calling an action like .show() or .write()), does Spark run the whole sequence at once.

### Wrapping Up
Today we built our first transformations in Spark. You learned how to:

- Use select() to pick and manipulate columns.
- Use filter() to keep only the rows you want.
- Use withColumn() to add or transform columns.
- Chain transformations into pipelines.

This is the bread and butter of Spark programming. From now on, every DataFrame pipeline you write will rely heavily on these three methods.

That’s Day 7. You’ve officially written your first Spark transformations.

Next up, Day 8: Actions in Spark — show, collect, count. We’ll explore how Spark actually executes the plan you’ve been building lazily until now.